# Clinical Notes NLP Pipeline

An end-to-end NLP pipeline for parsing, structuring, analysing, and querying free-text synthetic clinical notes (JSONL format).

## What this notebook covers

| Step | Description |
|------|-------------|
| 1. Data Loading | Read clinical notes from a JSONL file into a Pandas DataFrame |
| 2. Section Extraction | Regex-based parser to extract structured fields (HPI, LABS, PMH, ROS…) |
| 3. Lab Parsing | Parse LABS section into structured key-value pairs and wide columns |
| 4. Feature Engineering | Extract demographics (age, sex) and normalise lab values |
| 5. Patient-Level Summary | Aggregate to patient-level (HbA1c trends, HIV result, note count) |
| 6. Negation Detection | Rule-based regex negation patterns with unit tests |
| 7. RAG Pipeline | Embed note chunks into FAISS, retrieve by semantic similarity, answer with GPT-4o |
| 8. Agentic RAG | Tool-calling agent (GPT-4o-mini) that decides what to search before answering |

> ⚠️ All clinical data used in this notebook is entirely **synthetic and fictional**. Do not use for clinical decision-making.


## 1. Imports

In [30]:
# Uncomment to install dependencies (run once)
# %pip install pandas numpy faiss-cpu sentence-transformers openai python-dotenv

import json
import logging
import os
import re
import warnings
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from sentence_transformers import SentenceTransformer

pd.set_option("display.max_columns", None)
warnings.filterwarnings("ignore")
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

load_dotenv()  # reads OPENAI_API_KEY from .env
client = OpenAI()


## 2. Data Loading

Load the clinical notes from a JSONL file. Each line is one JSON object representing a single clinical note.  
Update `DATA_PATH` to point to your local copy of the data.

In [2]:
DATA_PATH = Path("data/clinical_notes.jsonl")  # <-- update this path

In [3]:
records = []
with open(DATA_PATH, "r", encoding="utf-8", errors="replace") as f:
    for i, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError as e:
            raise ValueError(f"Bad JSON on line {i}: {e}")

df = pd.DataFrame(records)
print(f"Loaded {df.shape[0]:,} notes | {df.shape[1]} columns")
df.head(2)


Loaded 364 notes | 6 columns


,note_id,patient_id,encounter_id,note_date,note_type,text
0,N0000001,P0001,E000001,2024-03-02 03:00:00,Discharge Summary,PATIENT: P0001 ENCOUNTER: E000001 AGE: 82 S...
1,N0000002,P0002,E000002,2025-11-22 05:00:00,ED Note,PATIENT: P0002 ENCOUNTER: E000002 AGE: 28 S...


## 3. Section Extraction

Clinical notes are unstructured free text. This step uses regex to automatically detect section headers (e.g. `HPI:`, `ALLERGIES:`, `Assessment/Plan:`) and extract their content into structured columns.

The approach:
1. **`find_headers`** — scans each note for lines matching `Header:` patterns and returns a deduplicated list.
2. **`extract_header_values`** — for each header, captures all text until the next header using a regex lookahead.
3. Headers are discovered **globally** across all notes to ensure consistent columns across the dataset.

In [4]:
def find_headers(note: str) -> list[str]:
    """
    Detect section headers in a clinical note.

    Matches patterns like 'Chief Complaint:', 'LABS:', 'Assessment/Plan:'
    at the start of a line. Returns a sorted, deduplicated list of header names.
    """
    if pd.isna(note) or not note:
        return []
    headers = re.findall(r"(?m)^\s*([A-Za-z][A-Za-z0-9 /&\-\(\)]*?)\s*:", note)
    return sorted({h.strip() for h in headers})


### 3a. Inspect headers in a sample note

In [5]:
# Inspect the headers detected in the first note
sample_note = df.loc[0, "text"]
sample_headers = find_headers(sample_note)
print(f"Headers found in note 0 ({len(sample_headers)} total):")
print(sample_headers)


Headers found in note 0 (11 total):
['ALLERGIES', 'Assessment/Plan', 'Chief Complaint', 'Counseling', 'HPI', 'LABS', 'Narrative', 'PATIENT', 'PE', 'PMH', 'ROS']


In [6]:
def extract_header_values(note: str, headers: list[str]) -> dict:
    """
    Extract section content for each header in a clinical note.

    For each header, captures all text following 'Header:' until the next
    known header or end of string, using a regex lookahead.

    Returns a dict mapping header name → extracted text (or None if absent).
    """
    if pd.isna(note) or not note:
        return {h: None for h in headers}

    headers_alt = "|".join(re.escape(h) for h in headers)
    out = {}
    for h in headers:
        pat = rf"(?is)(?m)^\s*{re.escape(h)}\s*:\s*(.*?)(?=^\s*(?:{headers_alt})\s*:|\Z)"
        m = re.search(pat, note)
        out[h] = m.group(1).strip() if m else None
    return out


In [7]:
# Normalise line endings for reliable line-start anchors
df["text_clean"] = (
    df["text"]
    .fillna("")
    .str.replace("\r\n", "\n", regex=False)
    .str.replace("\r", "\n", regex=False)
)

# Discover headers globally across all notes for consistent columns
all_headers: set[str] = set()
for note in df["text_clean"]:
    all_headers.update(find_headers(note))
headers = sorted(all_headers)
print(f"Discovered {len(headers)} unique section headers across all notes.")

# Extract sections and attach as new columns
sections = df["text_clean"].apply(lambda s: extract_header_values(s, headers)).apply(pd.Series)
df = pd.concat([df, sections], axis=1)
print(f"DataFrame shape after section extraction: {df.shape}")


Discovered 17 unique section headers across all notes.
DataFrame shape after section extraction: (364, 24)


In [8]:
# Validate section extraction on a sample note
SAMPLE_ID = "N0000001"
KEY_SECTIONS = ["Chief Complaint", "HPI", "LABS", "Assessment/Plan", "PMH", "ROS", "PE", "ALLERGIES"]

print(f"Section extraction validation for note {SAMPLE_ID}\n" + "-" * 50)
for section in KEY_SECTIONS:
    if section in df.columns:
        val = df.loc[df['note_id'] == SAMPLE_ID, section].iloc[0]
        preview = str(val)[:120].replace('\n', ' ') if val else 'NOT FOUND'
        print(f"{section:20s}: {preview}")


Section extraction validation for note N0000001
--------------------------------------------------
Chief Complaint     : anxiety and sleep disturbance
HPI                 : Presents for routine follow-up. Reports medication adherence is variable. Denies hypoglycemia.
LABS                : - HbA1c: 6.4% - WBC: 12.2 K/uL - Creatinine: 0.63 mg/dL - HIV Ag/Ab: pending - Gonorrhea NAAT: negative - Chlamydia NAAT
Assessment/Plan     : 1) Suspected UTI. UA/UCx ordered. Start nitrofurantoin. 2) Hydration encouraged.
PMH                 : HIV on ART; last viral load undetectable per patient.
ROS                 : Denies penile discharge. No testicular pain.
PE                  : GU exam deferred. No acute distress.
ALLERGIES           : penicillin (rash)


## 4. Lab Parsing

The `LABS` section contains bullet-point key-value pairs like `- HbA1c: 6.4%`.  
`parse_labs_block` converts this free text into a structured dict, which is then expanded into individual `lab_*` columns for downstream analysis.

In [9]:
def parse_labs_block(labs_text: str) -> dict:
    """
    Parse a free-text LABS section into a structured dict.

    Handles formats like:
        - HbA1c: 6.4%
        - WBC: 12.2 K/uL

    Returns {test_name: value_string} for each lab found.
    """
    if pd.isna(labs_text) or not str(labs_text).strip():
        return {}

    labs_text = str(labs_text).replace("\r\n", "\n").replace("\r", "\n")
    labs = {}
    for line in labs_text.split("\n"):
        line = line.strip()
        if not line:
            continue
        m = re.match(r"^-?\s*(.+?)\s*:\s*(.+)\s*$", line)
        if m:
            labs[m.group(1).strip()] = m.group(2).strip()
    return labs


In [10]:
df["labs_dict"] = df["LABS"].apply(parse_labs_block)

# Expand dict into wide lab columns
labs_wide = df["labs_dict"].apply(pd.Series).add_prefix("lab_")
df = pd.concat([df, labs_wide], axis=1)

print(f"Added {labs_wide.shape[1]} lab columns. Sample:")
print([c for c in df.columns if c.startswith("lab_")][:10])


Added 6 lab columns. Sample:
['lab_HbA1c', 'lab_WBC', 'lab_Creatinine', 'lab_HIV Ag/Ab', 'lab_Gonorrhea NAAT', 'lab_Chlamydia NAAT']


## 5. Feature Engineering

Extract structured demographic features (age, sex) from the PATIENT field using regex,  
and convert key lab values to numeric for downstream analysis.

In [11]:
# Extract age and sex from the PATIENT field
df[["age", "sex"]] = df["PATIENT"].str.extract(
    r"\bAGE:\s*(\d+)\s+SEX:\s*([A-Za-z])\b", expand=True
)
df["age"] = pd.to_numeric(df["age"], errors="coerce")

# Convert key lab columns to numeric (strip units e.g. '%', 'K/uL')
LAB_COLS = ["lab_HbA1c", "lab_WBC", "lab_Creatinine"]
for col in LAB_COLS:
    if col in df.columns:
        df[col] = pd.to_numeric(
            df[col].replace(r"[^0-9.\-]+", "", regex=True), errors="coerce"
        )

print(f"Age range: {df['age'].min():.0f} – {df['age'].max():.0f}")
print(f"Sex distribution:\n{df['sex'].value_counts()}")


Age range: 18 – 85
Sex distribution:
sex
M    143
F    139
Name: count, dtype: int64


### 5a. Normalise column names

Lowercase all columns and rename those with spaces or special characters to snake_case.

In [12]:
# Lowercase all column names
df.columns = df.columns.str.lower()

# Rename columns with spaces or special characters
RENAME_MAP = {
    "chief complaint": "chief_complaint",
    "assessment/plan": "assessment_plan",
    "lab_hba1c": "hba1c",
    "patient statement": "patient_statement",
    "quoted content": "quoted_content",
    "radiology report": "radiology_report",
}
df = df.rename(columns={k: v for k, v in RENAME_MAP.items() if k in df.columns})

print(f"Final DataFrame: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head(2)


Final DataFrame: 364 rows × 33 columns


,note_id,patient_id,encounter_id,note_date,note_type,text,text_clean,allergies,assessment_plan,chief_complaint,comment,counseling,findings,hpi,impression,labs,narrative,patient,pe,pmh,patient_statement,quoted_content,radiology_report,ros,labs_dict,hba1c,lab_wbc,lab_creatinine,lab_hiv ag/ab,lab_gonorrhea naat,lab_chlamydia naat,age,sex
0,N0000001,P0001,E000001,2024-03-02 03:00:00,Discharge Summary,PATIENT: P0001 ENCOUNTER: E000001 AGE: 82 S...,PATIENT: P0001 ENCOUNTER: E000001 AGE: 82 S...,penicillin (rash),1) Suspected UTI. UA/UCx ordered. Start nitrof...,anxiety and sleep disturbance,None,Encouraged routine testing and follow-up. Disc...,None,Presents for routine follow-up. Reports medica...,None,- HbA1c: 6.4%\n- WBC: 12.2 K/uL\n- Creatinine:...,The patient’s history is notable for intermitt...,P0001 ENCOUNTER: E000001 AGE: 82 SEX: F,GU exam deferred. No acute distress.,HIV on ART; last viral load undetectable per p...,None,None,None,Denies penile discharge. No testicular pain.,"{'HbA1c': '6.4%', 'WBC': '12.2 K/uL', 'Creatin...",6.4,12.2,0.63,pending,negative,positive,82.0,F
1,N0000002,P0002,E000002,2025-11-22 05:00:00,ED Note,PATIENT: P0002 ENCOUNTER: E000002 AGE: 28 S...,PATIENT: P0002 ENCOUNTER: E000002 AGE: 28 S...,sulfa drugs (hives),1) Suspected UTI. UA/UCx ordered. Start nitrof...,dysuria and urinary frequency,None,Provided education; patient expressed mistrust...,None,Reports mild pelvic discomfort. Denies vaginal...,None,- HbA1c: 6.1%\n- WBC: 7.7 K/uL\n- Creatinine: ...,The patient’s history is notable for intermitt...,P0002 ENCOUNTER: E000002 AGE: 28 SEX: F,Mild pharyngeal erythema. No exudates. No cerv...,None significant.,None,None,None,"Denies chest pain, denies SOB, denies nausea/v...","{'HbA1c': '6.1%', 'WBC': '7.7 K/uL', 'Creatini...",6.1,7.7,1.79,non-reactive,pending,positive,28.0,F


## 6. Patient-Level Summary

Aggregate note-level data up to the patient level:  
total note count, last visit date, maximum HbA1c across all notes, and most recent HIV Ag/Ab result.

In [13]:
df["note_date"] = pd.to_datetime(df["note_date"], errors="coerce")
df["hba1c"] = pd.to_numeric(df["hba1c"], errors="coerce")

# Normalise HIV result column
HIV_COL = "lab_hiv ag/ab"
if HIV_COL in df.columns:
    df[HIV_COL] = (
        df[HIV_COL]
        .astype(str).str.strip().str.lower()
        .replace({"nan": pd.NA, "none": pd.NA, "": pd.NA})
    )
    most_recent_hiv = (
        df.dropna(subset=[HIV_COL])
          .sort_values(["patient_id", "note_date"])
          .groupby("patient_id")[HIV_COL]
          .last()
    )
else:
    most_recent_hiv = pd.Series(dtype="object")

# Aggregate to patient level
patient_summary = (
    df.groupby("patient_id", as_index=False)
      .agg(
          n_notes=("note_id", "count"),
          last_note_date=("note_date", "max"),
          max_hba1c=("hba1c", "max"),
      )
)
patient_summary["most_recent_hiv_result"] = patient_summary["patient_id"].map(most_recent_hiv)

print(f"Patient summary: {patient_summary.shape[0]:,} patients")
patient_summary.head()


Patient summary: 60 patients


,patient_id,n_notes,last_note_date,max_hba1c,most_recent_hiv_result
0,P0001,1,2024-03-02 03:00:00,6.4,pending
1,P0002,9,2026-02-20 12:00:00,10.8,non-reactive
2,P0003,3,2025-12-16 14:00:00,10.4,positive
3,P0004,7,2025-11-23 20:00:00,10.2,non-reactive
4,P0005,2,2024-06-13 07:00:00,9.8,pending


## 7. Negation Detection

Rule-based regex patterns to detect negated symptoms in free-text clinical notes (e.g. *'denies fever'*, *'no chest pain'*).  
This is a lightweight alternative to full NLP negation frameworks (like NegEx) and is well-suited for structured synthetic data.

Negation flags are computed at the **note level**, then aggregated to **patient level** as counts.

In [14]:
NEG_PATTERNS = {
    "denies_fever":      r"\bdenies\b.*?\bfever\b|\bno\b.*?\bfever\b",
    "denies_chest_pain": r"\bdenies\b.*?\bchest pain\b|\bno\b.*?\bchest pain\b",
    "denies_sob":        r"\bdenies\b.*?\b(?:shortness of breath|sob)\b|\bno\b.*?\b(?:shortness of breath|sob)\b",
    "denies_dysuria":    r"\bdenies\b.*?\bdysuria\b|\bno\b.*?\bdysuria\b",
}

# Note-level boolean flags
for col, pat in NEG_PATTERNS.items():
    df[col] = df["text_clean"].fillna("").str.contains(pat, flags=re.I, regex=True)

# Aggregate to patient level
neg_counts = (
    df.groupby("patient_id", as_index=False)[list(NEG_PATTERNS.keys())].sum()
)
patient_summary = patient_summary.merge(neg_counts, on="patient_id", how="left")
patient_summary[list(NEG_PATTERNS.keys())] = (
    patient_summary[list(NEG_PATTERNS.keys())].fillna(0).astype(int)
)

print("Negation counts added to patient_summary:")
patient_summary[list(NEG_PATTERNS.keys())].describe().loc[["mean", "max"]]


Negation counts added to patient_summary:


,denies_fever,denies_chest_pain,denies_sob,denies_dysuria
mean,1.916667,1.983333,1.983333,1.85
max,6.000000,6.000000,6.000000,5.00


## 8. Negation Pattern Validation

Unit-test the negation patterns against known sentences to confirm true positives and true negatives before applying at scale.

In [15]:
FEVER_PAT = re.compile(r"\bdenies\b.*?\bfever\b|\bno\b.*?\bfever\b", flags=re.I)

TEST_CASES = [
    ("HPI: Patient denies fever.",                   True),
    ("No fever or chills.",                          True),
    ("Denies any fever, nausea, vomiting.",          True),
    ("Patient is feverish today.",                   False),  # should NOT match
    ("Denies chest pain. No shortness of breath.",   False),  # no fever mention
]

all_passed = True
for text, expected in TEST_CASES:
    result = bool(FEVER_PAT.search(text))
    status = "✅ PASS" if result == expected else "❌ FAIL"
    if result != expected:
        all_passed = False
    print(f"{status} | expected={expected} got={result} | {text!r}")

print("\nAll tests passed!" if all_passed else "\n⚠️  Some tests failed — review patterns.")


✅ PASS | expected=True got=True | 'HPI: Patient denies fever.'
✅ PASS | expected=True got=True | 'No fever or chills.'
✅ PASS | expected=True got=True | 'Denies any fever, nausea, vomiting.'
✅ PASS | expected=False got=False | 'Patient is feverish today.'
✅ PASS | expected=False got=False | 'Denies chest pain. No shortness of breath.'

All tests passed!


In [16]:
def inspect_negation_matches(series: pd.Series, pattern: str, n_samples: int = 5, window: int = 40) -> None:
    """Print match snippets with surrounding context for a given negation pattern."""
    compiled = re.compile(pattern, flags=re.I)
    sample = series.dropna().sample(min(n_samples, len(series)), random_state=42)
    for i, text in enumerate(sample, 1):
        matches = [
            text[max(0, m.start() - window): m.end() + window]
            for m in compiled.finditer(text)
        ]
        print(f"\n--- Sample {i} | {len(matches)} match(es) ---")
        for snippet in matches:
            print(" ", snippet.replace("\n", " "))

inspect_negation_matches(df["text_clean"], NEG_PATTERNS["denies_fever"])



--- Sample 1 | 1 match(es) ---
  ALLERGIES: shellfish (anaphylaxis) ROS: Denies dysuria. No flank pain. No fever/chills. PE: Anxious affect. Oriented x3

--- Sample 2 | 1 match(es) ---
  r. PMH: HTN, T2DM. ALLERGIES: NKDA ROS: Denies dysuria. No flank pain. No fever/chills. PE: Vitals stable. Lungs clear 

--- Sample 3 | 0 match(es) ---

--- Sample 4 | 0 match(es) ---

--- Sample 5 | 1 match(es) ---
  nt reports symptoms started 3 days ago. Denies fever. No chest pain. No shortness of breath.


## 9. RAG Pipeline

The RAG pipeline converts each note section into a searchable chunk, embeds them with `all-MiniLM-L6-v2`, stores them in a FAISS index, and uses GPT-4o to produce grounded answers from the retrieved context.

| Sub-step | Description |
|----------|-------------|
| 9a. Chunking | Split each note into section-level chunks with metadata |
| 9b. Embedding | Encode chunks with MiniLM-L6-v2 (384-dim, L2-normalised) |
| 9c. FAISS Index | Build and persist an IndexFlatIP (cosine similarity) index |
| 9d. Retrieval | Semantic search returning top-k chunks for a query |
| 9e. Generation | GPT-4o answers grounded only in retrieved context |
| 9f. Smoke tests | Validate end-to-end on example clinical questions |

### 9a. Build Note Chunks

Each row in `chunks_df` represents one searchable unit: a single section from a single note, with its metadata (note_id, patient_id, date, section name) preserved for citation in answers.

In [17]:
def build_chunks(df: pd.DataFrame, text_col: str = "text_clean") -> pd.DataFrame:
    """
    Split each note into section-level chunks.

    Each chunk contains:
      - note_id, patient_id, note_date  (metadata for citation)
      - section                         (header name, e.g. 'HPI')
      - chunk_text                      (section content)

    Notes with no detected sections are kept as a single full-text chunk.
    """
    rows = []
    for _, row in df.iterrows():
        note = str(row.get(text_col, "") or "")
        detected = find_headers(note)

        if not detected:
            # No section headers — keep the whole note as one chunk
            rows.append({
                "note_id":    row.get("note_id"),
                "patient_id": row.get("patient_id"),
                "note_date":  row.get("note_date"),
                "section":    "FULL_NOTE",
                "chunk_text": note.strip(),
            })
            continue

        extracted = extract_header_values(note, detected)
        for section, content in extracted.items():
            if content and content.strip():
                rows.append({
                    "note_id":    row.get("note_id"),
                    "patient_id": row.get("patient_id"),
                    "note_date":  row.get("note_date"),
                    "section":    section,
                    "chunk_text": content.strip(),
                })
    return pd.DataFrame(rows)


chunks_df = build_chunks(df)
print(f"Built {len(chunks_df):,} chunks from {df.shape[0]:,} notes.")
print(f"Sections per note (avg): {len(chunks_df) / df.shape[0]:.1f}")
chunks_df.head(3)


Built 3,308 chunks from 364 notes.
Sections per note (avg): 9.1


,note_id,patient_id,note_date,section,chunk_text
0,N0000001,P0001,2024-03-02 03:00:00,ALLERGIES,penicillin (rash)
1,N0000001,P0001,2024-03-02 03:00:00,Assessment/Plan,1) Suspected UTI. UA/UCx ordered. Start nitrof...
2,N0000001,P0001,2024-03-02 03:00:00,Chief Complaint,anxiety and sleep disturbance


### 9b. Embed Chunks

Encode every chunk with `sentence-transformers/all-MiniLM-L6-v2` (384-dimensional, L2-normalised vectors). Normalisation enables cosine similarity via inner product, which FAISS `IndexFlatIP` computes efficiently.

In [18]:
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
INDEX_PATH       = "rag_chunks.index"
CHUNKS_PATH      = "rag_chunks.json"

embed_model = SentenceTransformer(EMBED_MODEL_NAME)

texts = chunks_df["chunk_text"].tolist()
embeddings = embed_model.encode(
    texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype("float32")

print(f"Embeddings shape: {embeddings.shape}")


Batches: 100%|██████████| 104/104 [00:22<00:00,  4.55it/s]

Embeddings shape: (3308, 384)


### 9c. Build & Persist FAISS Index

In [19]:
# Build FAISS index (inner product = cosine sim on L2-normalised vectors)
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

# Persist index and chunk records so they can be reloaded without re-embedding
faiss.write_index(index, INDEX_PATH)
chunks_df.to_json(CHUNKS_PATH, orient="records", force_ascii=False, indent=2)

print(f"FAISS index saved  → {INDEX_PATH}  ({index.ntotal:,} vectors)")
print(f"Chunk records saved → {CHUNKS_PATH}")


FAISS index saved  → rag_chunks.index  (3,308 vectors)
Chunk records saved → rag_chunks.json


### 9d. Retrieval

`retrieve` encodes the query, searches the FAISS index, and returns the top-k most similar chunks as a DataFrame ready to pass into the prompt builder.

In [20]:
def retrieve(question: str, k: int = 5) -> pd.DataFrame:
    """
    Retrieve the top-k note chunks most semantically similar to *question*.

    Args:
        question: The clinical question to search for.
        k:        Number of chunks to return.

    Returns:
        DataFrame with columns: note_id, patient_id, note_date, section,
        chunk_text, score.
    """
    q_vec = embed_model.encode(
        [question], normalize_embeddings=True
    ).astype("float32")
    scores, ids = index.search(q_vec, k)

    results = chunks_df.iloc[ids[0]].copy()
    results["score"] = scores[0]
    return results.reset_index(drop=True)


# Quick retrieval smoke-test
test_hits = retrieve("What is the patient's HbA1c result?")
print(test_hits[["note_id", "patient_id", "section", "score"]].to_string(index=False))


 note_id patient_id section    score
N0000058      P0013    LABS 0.555226
N0000143      P0026    LABS 0.540859
N0000322      P0053    LABS 0.540530
N0000240      P0039    LABS 0.539732
N0000158      P0028    LABS 0.538698


### 9e. Generation

`generate_answer` retrieves the most relevant chunks and passes them to GPT-4o with a strict instruction to answer only from the context and return structured JSON evidence.

In [21]:
RAG_SYSTEM_PROMPT = (
    "You are a clinical data abstraction assistant. "
    "Answer the QUESTION using ONLY the CONTEXT provided. "
    "If the answer is not explicitly stated in the context, reply UNKNOWN. "
    "Return a JSON object with exactly two keys:\n"
    "  - answer: a concise string answer\n"
    "  - evidence: a list of objects, each with note_id, date, section, "
    "and a short supporting quote from the context."
)


def build_rag_prompt(question: str, retrieved: pd.DataFrame) -> str:
    """
    Build the user-turn prompt from retrieved chunks.

    Each chunk is formatted with its metadata header so the model can
    cite specific notes in its evidence output.
    """
    context_blocks = [
        f"NOTE_ID: {r['note_id']} | PATIENT_ID: {r['patient_id']} "
        f"| DATE: {r['note_date']} | SECTION: {r['section']}\n{r['chunk_text']}"
        for _, r in retrieved.iterrows()
    ]
    context = "\n\n---\n\n".join(context_blocks)
    return f"QUESTION:\n{question}\n\nCONTEXT:\n{context}"


def generate_answer(question: str, k: int = 5, max_tokens: int = 400) -> dict:
    """
    Full RAG pipeline: retrieve → prompt → GPT-4o → parsed JSON answer.

    Returns a dict with 'answer' and 'evidence' keys, or an error message.
    """
    retrieved = retrieve(question, k=k)
    user_prompt = build_rag_prompt(question, retrieved)

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": RAG_SYSTEM_PROMPT},
            {"role": "user",   "content": user_prompt},
        ],
        max_tokens=max_tokens,
        temperature=0,
        response_format={"type": "json_object"},
    )
    raw = response.choices[0].message.content
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"answer": raw, "evidence": []}


### 9f. Smoke Tests

In [22]:
TEST_QUESTIONS = [
    "What is the HbA1c result?",
    "What is the HIV Ag/Ab result?",
    "What is the assessment and plan?",
    "Does the patient deny fever?",
    "What are the patient's allergies?",
]

for q in TEST_QUESTIONS:
    result = generate_answer(q)
    print(f"Q: {q}")
    print(f"A: {result.get('answer', 'N/A')}")
    evidence = result.get("evidence", [])
    if evidence:
        e = evidence[0]
        print(f"   ↳ {e.get('note_id')} | {e.get('section')} | {str(e.get('quote',''))[:80]}")
    print()


Q: What is the HbA1c result?
A: 8.3%
   ↳ N0000058 | LABS | HbA1c: 8.3%

Q: What is the HIV Ag/Ab result?
A: pending
   ↳ N0000034 | LABS | HIV Ag/Ab: pending

Q: What is the assessment and plan?
A: UNKNOWN

Q: Does the patient deny fever?
A: Yes
   ↳ N0000044 | HPI | Denies fever.

Q: What are the patient's allergies?
A: seasonal allergies
   ↳ N0000057 | PMH | Anxiety, seasonal allergies.



## 10. Agentic RAG

The baseline RAG always runs the same fixed retrieve-then-generate loop. The **agent** adds decision-making: the model is given a set of tools and decides *which* to call and *what* to search for before producing its final answer.

### Tools available to the agent

| Tool | Description |
|------|-------------|
| `search_notes` | Semantic search over all note chunks (returns top-k with metadata) |
| `get_patient_summary` | Look up the structured patient-level summary (HbA1c, HIV, negation flags) |
| `get_all_sections_for_patient` | Retrieve every section chunk for a specific patient |

Multiple tools allow the agent to choose the most appropriate retrieval strategy per question — something a fixed pipeline cannot do.

The loop follows the standard two-turn Chat Completions tool-call pattern:
1. First call → model emits `tool_calls`.
2. Tools are executed locally.
3. Second call → model reads results and returns the final answer.

### 10a. Tool Definitions & Implementations

In [23]:
AGENT_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_notes",
            "description": (
                "Semantically search all clinical note chunks and return the most "
                "relevant ones for the given question."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "question": {
                        "type": "string",
                        "description": "The clinical question to search for.",
                    },
                    "k": {
                        "type": "integer",
                        "description": "Number of chunks to retrieve (default 5).",
                    },
                },
                "required": ["question"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_patient_summary",
            "description": (
                "Return the structured patient-level summary for a given patient_id, "
                "including note count, last visit date, max HbA1c, HIV result, "
                "and negation symptom counts."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "patient_id": {
                        "type": "string",
                        "description": "The patient identifier (e.g. 'P0000001').",
                    },
                },
                "required": ["patient_id"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_all_sections_for_patient",
            "description": (
                "Retrieve every section chunk from all notes for a specific patient. "
                "Use this when you need a comprehensive view of a patient's full history."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "patient_id": {
                        "type": "string",
                        "description": "The patient identifier.",
                    },
                },
                "required": ["patient_id"],
                "additionalProperties": False,
            },
        },
    },
]


In [24]:
# ── Tool implementations ────────────────────────────────────────────────────

def tool_search_notes(question: str, k: int = 5) -> list[dict]:
    """Return top-k semantically similar chunks for *question*."""
    hits = retrieve(question, k=k)
    return hits[["note_id", "patient_id", "note_date", "section", "chunk_text", "score"]]\
        .to_dict(orient="records")


def tool_get_patient_summary(patient_id: str) -> dict:
    """Return the patient-level summary row for *patient_id* as a dict."""
    row = patient_summary[patient_summary["patient_id"] == patient_id]
    if row.empty:
        return {"error": f"Patient '{patient_id}' not found in summary."}
    return row.iloc[0].to_dict()


def tool_get_all_sections_for_patient(patient_id: str) -> list[dict]:
    """Return all section chunks for *patient_id*, sorted by date and section."""
    rows = chunks_df[chunks_df["patient_id"] == patient_id]\
        .sort_values(["note_date", "section"])
    if rows.empty:
        return [{"error": f"No chunks found for patient '{patient_id}'."}]
    return rows[["note_id", "note_date", "section", "chunk_text"]].to_dict(orient="records")


TOOL_DISPATCH = {
    "search_notes":                  tool_search_notes,
    "get_patient_summary":           tool_get_patient_summary,
    "get_all_sections_for_patient":  tool_get_all_sections_for_patient,
}

print("Tools registered:", list(TOOL_DISPATCH.keys()))


Tools registered: ['search_notes', 'get_patient_summary', 'get_all_sections_for_patient']


### 10b. Agent Loop

In [25]:
AGENT_SYSTEM_PROMPT = (
    "You are a clinical data abstraction assistant with access to a database of "
    "structured clinical notes. "
    "Use the available tools to find the information needed before answering. "
    "Always answer ONLY from tool results — never from prior knowledge. "
    "Return a JSON object with:\n"
    "  - answer: a concise string\n"
    "  - evidence: list of objects with note_id, date, section, and a short quote.\n"
    "If the answer cannot be found in the tool results, return UNKNOWN as the answer."
)


def agent_answer(
    question: str,
    model_name: str = "gpt-4o-mini",
    max_tokens: int = 500,
    max_tool_rounds: int = 3,
) -> dict:
    """
    Answer *question* using an agentic tool-calling loop.

    The model iteratively calls tools (search_notes, get_patient_summary,
    get_all_sections_for_patient) until it has enough information to answer,
    or until max_tool_rounds is reached.

    Args:
        question:        The clinical question to answer.
        model_name:      OpenAI model to use.
        max_tokens:      Maximum tokens for each completion call.
        max_tool_rounds: Maximum number of tool-call rounds before forcing a final answer.

    Returns:
        Parsed JSON dict with 'answer' and 'evidence' keys.
    """
    messages = [
        {"role": "system", "content": AGENT_SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]

    for round_num in range(max_tool_rounds):
        response = client.chat.completions.create(
            model=model_name,
            messages=messages,
            tools=AGENT_TOOLS,
            tool_choice="auto",
            max_tokens=max_tokens,
            temperature=0,
            response_format={"type": "json_object"} if round_num > 0 else None,
        )

        msg = response.choices[0].message
        finish = response.choices[0].finish_reason

        # No tool calls → model is ready to give the final answer
        if finish == "stop" or not msg.tool_calls:
            raw = msg.content or "{}"
            try:
                return json.loads(raw)
            except json.JSONDecodeError:
                return {"answer": raw, "evidence": []}

        # Execute each tool call and append results to message history
        messages.append(msg)
        for call in msg.tool_calls:
            fn_name = call.function.name
            fn_args = json.loads(call.function.arguments)

            if fn_name not in TOOL_DISPATCH:
                result = {"error": f"Unknown tool: {fn_name}"}
            else:
                result = TOOL_DISPATCH[fn_name](**fn_args)

            messages.append({
                "role":         "tool",
                "tool_call_id": call.id,
                "content":      json.dumps(result, default=str),
            })

    # max_tool_rounds reached — force a final answer
    messages.append({
        "role":    "user",
        "content": "Based on the tool results above, provide your final JSON answer now.",
    })
    final = client.chat.completions.create(
        model=model_name,
        messages=messages,
        max_tokens=max_tokens,
        temperature=0,
        response_format={"type": "json_object"},
    )
    raw = final.choices[0].message.content or "{}"
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"answer": raw, "evidence": []}


### 10c. Agent Smoke Tests

These questions test different tool paths:
- Questions 1–2 use `search_notes` (semantic similarity across all notes)
- Question 3 uses `get_patient_summary` (structured aggregated data for a specific patient)
- Question 4 uses `get_all_sections_for_patient` for a full patient history lookup

In [28]:
AGENT_TEST_QUESTIONS = [
    "What is the most recent HbA1c result across all notes?",
    "What lab results are recorded for patient P0003?",
    "How many notes does patient P0002 have and what is their max HbA1c?",
    "Summarise the assessment and plan for patient P0001.",
]

for q in AGENT_TEST_QUESTIONS:
    print(f"{'='*60}")
    print(f"Q: {q}")
    result = agent_answer(q)
    print(f"A: {result.get('answer', 'N/A')}")
    evidence = result.get("evidence", [])
    for e in evidence[:2]:
        print(f"   ↳ {e.get('note_id','?')} | {e.get('section','?')} | {str(e.get('quote',''))[:80]}")
    print()


Q: What is the most recent HbA1c result across all notes?
A: 9.8%
   ↳ N0000021 | LABS | - HbA1c: 9.8%

Q: What lab results are recorded for patient P0003?
A: The lab results for patient P0003 include HbA1c: 10.4%, WBC: 4.4 K/uL, Creatinine: 1.64 mg/dL, HIV Ag/Ab: positive, Gonorrhea NAAT: negative, Chlamydia NAAT: negative.
   ↳ N0000013 | LABS | - HbA1c: 10.4%
- WBC: 4.4 K/uL
- Creatinine: 1.64 mg/dL
- HIV Ag/Ab: positive
- 

Q: How many notes does patient P0002 have and what is their max HbA1c?
A: Patient P0002 has 9 notes and a max HbA1c of 10.8.
   ↳ summary | summary | n_notes: 9, max_hba1c: 10.8

Q: Summarise the assessment and plan for patient P0001.
A: 1) Suspected UTI. UA/UCx ordered. Start nitrofurantoin. 2) Hydration encouraged.
   ↳ N0000001 | Assessment/Plan | 1) Suspected UTI. UA/UCx ordered. Start nitrofurantoin.
2) Hydration encouraged.



### 10d. Baseline vs Agent Comparison

Run the same question through both pipelines side by side to see the difference.

In [27]:
COMPARISON_QUESTION = "What is the assessment and plan for patient P0002?"

print("BASELINE RAG")
print("-" * 40)
baseline = generate_answer(COMPARISON_QUESTION)
print(f"Answer   : {baseline.get('answer')}")
print(f"Evidence : {len(baseline.get('evidence', []))} citations")
if baseline.get('evidence'):
    e = baseline['evidence'][0]
    print(f"   ↳ {e.get('note_id')} | {e.get('section')} | {str(e.get('quote',''))[:80]}")

print()
print("AGENTIC RAG")
print("-" * 40)
agent = agent_answer(COMPARISON_QUESTION)
print(f"Answer   : {agent.get('answer')}")
print(f"Evidence : {len(agent.get('evidence', []))} citations")
if agent.get('evidence'):
    e = agent['evidence'][0]
    print(f"   ↳ {e.get('note_id')} | {e.get('section')} | {str(e.get('quote',''))[:80]}")


BASELINE RAG
----------------------------------------
Answer   : UNKNOWN
Evidence : 0 citations

AGENTIC RAG
----------------------------------------
Answer   : 1) Suspected UTI. UA/UCx ordered. Start nitrofurantoin.
2) Hydration encouraged.
Evidence : 3 citations
   ↳ N0000005 | Assessment/Plan | 1) Suspected UTI. UA/UCx ordered. Start nitrofurantoin.
2) Hydration encouraged.


###  Summary
This notebook demonstrates a complete NLP and Retrieval-Augmented Generation pipeline built from scratch on 364 synthetic clinical notes across 60 patients. The pipeline begins with regex-based section extraction, automatically discovering 17 unique clinical headers (HPI, LABS, PMH, Assessment/Plan, etc.) directly from free text without any predefined schema. Lab results are then parsed from the LABS section into structured columns, and patient demographics are extracted using targeted regex patterns. At the patient level, notes are aggregated to track metrics like maximum HbA1c and most recent HIV Ag/Ab result across visits. A rule-based negation detection layer identifies symptom denials (fever, chest pain, shortness of breath, dysuria) with unit-tested regex patterns validated against known true positives and negatives.

The RAG pipeline converts all 364 notes into 3,308 section-level chunks — an average of 9.1 sections per note — encodes them with `sentence-transformers/all-MiniLM-L6-v2` into 384-dimensional vectors, and stores them in a FAISS index for fast cosine similarity search. GPT-4o answers questions grounded strictly in retrieved context and returns structured JSON with evidence citations tied to specific note IDs and sections. The agentic layer extends this with three tools — semantic search, patient summary lookup, and full patient history retrieval — giving GPT-4o-mini the ability to choose the right retrieval strategy per question. The comparison between baseline and agentic RAG on the same question illustrates the core difference: the baseline returns UNKNOWN because it has no concept of patient identity, while the agent correctly routes to the patient-specific tool and returns a cited, accurate answer.

---
*All patient data in this notebook is entirely synthetic and generated for NLP practice purposes only. Do not use for clinical decision-making.*